# Project: Danish town name generator 🏘️
Denmark has wonderful town names: *Hvide Sande, Tappernøje, Rødby, Sindal, Øster Assels*. Train **character-level language models** on all ~1,500 urban areas from Statistics Denmark (a bigram counting model, then a small neural net in the style of Karpathy's *makemore*) and invent new ones that sound right. Then check that they're actually new.

Topic: [[NLP Overview]], [[Text Representations]] · guide note: [[Project - Danish Town Name Generator]] · related: [[Mini-GPT on Danish]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, re, urllib.request, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

f = pathlib.Path("data/danish_towns.txt")
if not f.exists():
    f.parent.mkdir(exist_ok=True)
    info = json.load(urllib.request.urlopen("https://api.statbank.dk/v1/tableinfo/BY1?lang=da&format=JSON"))   # Statistics Denmark, table BY1
    names = set()
    for v in info["variables"][0]["values"]:
        t = re.sub(r"\s*\(.*?\)", "", re.sub(r"^[0-9-]+ ", "", v["text"])).strip()
        if not any(k in t for k in ("Landdistrikt", "Kommune", "Uden fast", "område", "Hele landet")): names.add(t)
    f.write_text("\n".join(sorted(names)), encoding="utf-8")
words = [w.lower().replace(".", "") for w in f.read_text(encoding="utf-8").split("\n") if w]   # "St. Heddinge" → "st heddinge" ("." is our start/end marker)
rng = np.random.default_rng(0); perm = rng.permutation(len(words))
train = [words[i] for i in perm[:int(.9 * len(words))]]; val = [words[i] for i in perm[int(.9 * len(words)):]]
chars = ["."] + sorted(set("".join(words)))       # "." marks start and end of a name
print(f"{len(words)} names, {len(chars)} symbols, e.g. {rng.choice(words, 8).tolist()}")

## 1. Bigram model
Count how often each character follows another (with `.` for start/end), add `alpha` to every count (smoothing), and normalise each row to probabilities. Return the matrix `P[current, next]`.

In [ ]:
def bigram_model(words, chars, alpha=1.0):
    # TODO 1: smoothed bigram counts over "." + word + "." → row-normalised probabilities
    raise NotImplementedError  # TODO 1

## 2. How good is a model? Negative log-likelihood
Average $-\log P(\text{next}\mid\text{current})$ over every character transition (including the final `.`) in a list of words. Lower is better; uniform guessing gives $\ln(\#\text{symbols})$.

In [ ]:
def nll(P, words, chars):
    # TODO 2: mean negative log-probability per transition
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_big():
    ch = [".", "a", "b"]; P = bigram_model(["ab"], ch, alpha=0.0)
    return np.allclose(P[0], [0, 1, 0]) and np.allclose(P[2], [1, 0, 0]) and np.isclose(nll(P, ["ab"], ch), 0) and \
        np.isclose(nll(bigram_model(["ab"], ch, alpha=1.0), ["ab"], ch), -np.mean(np.log([2 / 4, 2 / 4, 2 / 4])))
check("bigram", _t_big, "On ['ab'] without smoothing every transition is certain (NLL 0); with alpha=1 each is 2/4.")

def sample_bigram(P, n, seed=0):
    r = np.random.default_rng(seed); out = []
    for _ in range(n):
        i, s = 0, ""
        while True:
            i = r.choice(len(chars), p=P[i])
            if i == 0: break
            s += chars[i]
        out.append(s.title())
    return out
if ready("bigram"):
    P = bigram_model(train, chars, alpha=0.1)
    print(f"val NLL: uniform {np.log(len(chars)):.3f}, bigram {nll(P, val, chars):.3f}")
    print("bigram towns:", sample_bigram(P, 12))

## 3. A tiny neural model (makemore-style)
Look at the last **3** characters through a learned embedding, then an MLP. Same idea as Bengio et al. 2003. It runs as given; compare its validation NLL with the bigram.

In [ ]:
BLOCK = 3; stoi = {c: i for i, c in enumerate(chars)}
def dataset(ws):
    X, Y = [], []
    for w in ws:
        ctx = [0] * BLOCK
        for c in w + ".":
            X.append(ctx); Y.append(stoi[c]); ctx = ctx[1:] + [stoi[c]]
    return torch.tensor(X), torch.tensor(Y)
Xtr, Ytr = dataset(train); Xva, Yva = dataset(val)
model = nn.Sequential(nn.Embedding(len(chars), 16), nn.Flatten(), nn.Linear(16 * BLOCK, 128), nn.Tanh(), nn.Linear(128, len(chars)))
opt = torch.optim.AdamW(model.parameters(), 3e-3, weight_decay=0.01)
for step in range(4000):
    i = torch.randint(0, len(Xtr), (64,)); loss = F.cross_entropy(model(Xtr[i]), Ytr[i]); opt.zero_grad(); loss.backward(); opt.step()
model.eval()
with torch.no_grad(): mlp_val = F.cross_entropy(model(Xva), Yva).item()
print(f"val NLL MLP (3 chars of context): {mlp_val:.3f}")

def sample_mlp(n, temp=0.9, seed=0):
    g = torch.Generator().manual_seed(seed); out = []
    for _ in range(n):
        ctx, s = [0] * BLOCK, ""
        while len(s) < 25:
            with torch.no_grad(): p = F.softmax(model(torch.tensor([ctx])) / temp, -1)
            i = torch.multinomial(p, 1, generator=g).item()
            if i == 0: break
            s += chars[i]; ctx = ctx[1:] + [i]
        out.append(s.title())
    return out
towns = sample_mlp(30); known = set(words)
print("MLP towns:", towns)
print(f"{sum(t.lower() not in known for t in towns)}/30 are not in the training list of urban areas (some may still be real villages!)")

In [ ]:
if ready("bigram"):
    check("neural beats bigram", lambda: mlp_val < nll(P, val, chars) - 0.1, "With 3 characters of context the MLP should beat the bigram by > 0.1 nats.")

<details><summary>▶ Solution</summary>

```python
def bigram_model(words, chars, alpha=1.0):
    stoi = {c: i for i, c in enumerate(chars)}
    N = np.full((len(chars), len(chars)), alpha)
    for w in words:
        s = "." + w + "."
        for a, c in zip(s, s[1:]):
            N[stoi[a], stoi[c]] += 1
    return N / N.sum(1, keepdims=True)
```

```python
def nll(P, words, chars):
    stoi = {c: i for i, c in enumerate(chars)}
    total, n = 0.0, 0
    for w in words:
        s = "." + w + "."
        for a, c in zip(s, s[1:]):
            total -= np.log(P[stoi[a], stoi[c]]); n += 1
    return total / n
```
</details>

## Stretch goals
- Condition on the region: generate names that sound like Southern Jutland (*-bøl, -skov*) vs. Zealand (*-løse, -rup*).
- Increase `BLOCK` to 5 and the embedding to 32. When does it start memorising real names?
- Which endings does the model love? Count the last 3 letters of 1,000 samples vs. real names (*-rup, -by, -strup, -lev*).